# Task 3 — Jaccard Results Table (Qwen1.5-MoE)
**Three-model comparison: Phi-3.5-MoE | OLMoE-1B-7B | Qwen1.5-MoE-A2.7B**

Reads the output of `qwen-jaccard-diagnostic.ipynb` and formats the results
in the same comparison table used in the thesis.

**Run `qwen-jaccard-diagnostic.ipynb` first** — this notebook requires
`qwen_experiment_config.json` to exist in the working directory.

## What this produces
1. Three-way comparison table: Phi-3.5-MoE | OLMoE-1B-7B | Qwen1.5-MoE Jaccard scores
2. Architecture comparison: expert counts, routing, uniform baselines
3. Per-layer Jaccard table for Qwen with top-3 highest-overlap layers highlighted
4. Interpretation note for the thesis (Section 4.2)

In [1]:
import json
import os

# On Kaggle: adjust to where qwen_experiment_config.json was saved
# Either upload as a dataset input or use /kaggle/working/ if saved during this session
CONFIG_PATH = "/content/qwen_experiment_config.json"

if not os.path.exists(CONFIG_PATH):
    # Fallback: local working directory (if running locally or saved in same session)
    CONFIG_PATH = "qwen_experiment_config.json"

if not os.path.exists(CONFIG_PATH):
    raise FileNotFoundError(
        CONFIG_PATH + " not found.\n"
        "Run qwen-jaccard-diagnostic.ipynb first, download qwen_experiment_config.json, "
        "and upload it as a Kaggle dataset input."
    )

with open(CONFIG_PATH) as f:
    cfg = json.load(f)

print("Config loaded:")
print(json.dumps(cfg, indent=2))

Config loaded:
{
  "model_name": "Qwen/Qwen1.5-MoE-A2.7B",
  "model_dim": 2048,
  "ffn_dim": 1408,
  "shared_expert_ffn_dim": 5632,
  "num_layers": 24,
  "num_routing_experts": 60,
  "num_shared_experts": 1,
  "top_k": 4,
  "top_n_jaccard": 15,
  "jaccard_python_medical": 0.0714,
  "jaccard_math_creative": 0.1315,
  "jaccard_null_baseline": 0.5101,
  "chosen_option": "A",
  "chosen_pair": "Python vs Medical",
  "target_layer": 0,
  "target_expert": 5,
  "primary_dataset": "openai/openai_humaneval",
  "primary_col": "prompt",
  "primary_config": null,
  "conflict_dataset": "qiaojin/PubMedQA",
  "conflict_col": "question",
  "conflict_config": "pqa_labeled",
  "domain_names": [
    "code",
    "medical"
  ],
  "olmoe_jaccard_math_creative": 0.0528,
  "olmoe_null_baseline": 0.8403
}


## Table 1 — Three-Model Jaccard Comparison

In [2]:
# ── Reference values from existing thesis tables ──────────────────────────────
PHI_JACCARD_PYTHON_MEDICAL = 0.056   # Python vs Medical
PHI_JACCARD_MATH_CREATIVE  = 0.069   # Math vs Creative

OLMOE_JACCARD_PYTHON_MEDICAL = 0.1029
OLMOE_JACCARD_MATH_CREATIVE  = 0.0528
OLMOE_NULL                   = 0.8403
OLMOE_TOP_N                  = 16

# ── Qwen values from config ────────────────────────────────────────────────────
qwen_jaccard_a = cfg["jaccard_python_medical"]   # Python vs Medical
qwen_jaccard_b = cfg.get("jaccard_math_creative", None)  # may not exist if only A was run
qwen_null      = cfg["jaccard_null_baseline"]
qwen_top_n     = cfg.get("top_n_jaccard", 15)
qwen_top_k     = cfg.get("top_k", 4)
qwen_n_experts = cfg.get("num_experts", 60)
qwen_n_layers  = cfg.get("num_layers", 24)

# Uniform baseline: top_k / num_routing_experts
# Phi: 2/16 = 12.5% | OLMoE: 8/64 = 12.5% | Qwen: 4/60 ≈ 6.7%
qwen_uniform_pct = 100.0 * qwen_top_k / qwen_n_experts

def sep_status(score, null):
    ratio = score / max(null, 0.001)
    if ratio < 0.2:   return "Separated      (%.2fx)" % ratio
    elif ratio < 0.5: return "Moderate overlap (%.2fx)" % ratio
    else:             return "High overlap   (%.2fx)" % ratio

qwen_b_str = "%.3f" % qwen_jaccard_b if qwen_jaccard_b is not None else "N/A"
olmoe_b_status = sep_status(OLMOE_JACCARD_MATH_CREATIVE, OLMOE_NULL)
qwen_a_status  = sep_status(qwen_jaccard_a, qwen_null)
qwen_b_status  = sep_status(qwen_jaccard_b, qwen_null) if qwen_jaccard_b is not None else "N/A"

print("TABLE 1: Jaccard Routing Overlap — Three-Model Comparison")
print("=" * 90)

rows = [
    ("Domain Pair",        "Phi-3.5-MoE", "OLMoE-1B-7B", "Qwen1.5-MoE", "Status (Qwen)"),
    ("-" * 22,             "-" * 12,       "-" * 12,        "-" * 12,       "-" * 26),
    ("Python vs Medical",
     "%.3f" % PHI_JACCARD_PYTHON_MEDICAL,
     "%.3f" % OLMOE_JACCARD_PYTHON_MEDICAL,
     "%.3f" % qwen_jaccard_a,
     qwen_a_status),
    ("Math vs Creative",
     "%.3f" % PHI_JACCARD_MATH_CREATIVE,
     "%.3f" % OLMOE_JACCARD_MATH_CREATIVE,
     qwen_b_str,
     qwen_b_status),
    ("Null (same domain)",
     "~1.0",
     "%.3f" % OLMOE_NULL,
     "%.3f" % qwen_null,
     "baseline"),
]
for r in rows:
    print("%-24s %14s %14s %14s  %s" % r)

print()
print("TABLE 1b: Architecture Comparison")
print("-" * 65)
arch_rows = [
    ("Architecture",       "Phi-3.5-MoE", "OLMoE-1B-7B", "Qwen1.5-MoE"),
    ("-" * 22,             "-" * 12,       "-" * 12,        "-" * 12),
    ("Routing experts",    "16",           "64",            "%d" % qwen_n_experts),
    ("Shared expert",      "None",         "None",          "1 (always-on)"),
    ("Top-k routing",      "2",            "8",             "%d" % qwen_top_k),
    ("Uniform baseline",   "12.5%",        "12.5%",         "%.1f%%" % qwen_uniform_pct),
    ("Num layers",         "32",           "16",            "%d" % qwen_n_layers),
    ("Jaccard method",     "raw set",      "top-%d" % OLMOE_TOP_N, "top-%d" % qwen_top_n),
    ("Jaccard metric",     "routing only", "routing only",  "routing only"),
]
for r in arch_rows:
    print("%-24s %14s %14s %14s" % r)

print()
chosen_j = qwen_jaccard_a if cfg.get("chosen_option", "A") == "A" else (qwen_jaccard_b or qwen_jaccard_a)
chosen_ratio = chosen_j / max(qwen_null, 0.001)
print("Chosen pair: %s (Option %s)" % (cfg.get("chosen_pair", "Python vs Medical"), cfg.get("chosen_option", "A")))
print("  Top-K Jaccard:    %.3f" % chosen_j)
print("  Null baseline:    %.3f" % qwen_null)
print("  Separation ratio: %.2fx above null" % chosen_ratio)

TABLE 1: Jaccard Routing Overlap — Three-Model Comparison
Domain Pair                 Phi-3.5-MoE    OLMoE-1B-7B    Qwen1.5-MoE  Status (Qwen)
----------------------     ------------   ------------   ------------  --------------------------
Python vs Medical                 0.056          0.103          0.071  Separated      (0.14x)
Math vs Creative                  0.069          0.053          0.132  Moderate overlap (0.26x)
Null (same domain)                 ~1.0          0.840          0.510  baseline

TABLE 1b: Architecture Comparison
-----------------------------------------------------------------
Architecture                Phi-3.5-MoE    OLMoE-1B-7B    Qwen1.5-MoE
----------------------     ------------   ------------   ------------
Routing experts                      16             64             60
Shared expert                      None           None  1 (always-on)
Top-k routing                         2              8              4
Uniform baseline                  12.5

## Table 2 — Per-Layer Jaccard for Qwen (Top-3 Highest-Overlap Layers)

Requires `qwen_freq_cache.pkl` — saved by `qwen-jaccard-diagnostic.ipynb`.
If the cache is not available, the target layer and expert from the config
are reported instead.

**Note:** Cell 11 of the diagnostic notebook prints full per-layer scores to stdout.
Check that notebook's output for the complete breakdown.

In [3]:
import pickle

# Adjust to where qwen_freq_cache.pkl was saved/uploaded
FREQ_CACHE = "/kaggle/input/datasets/ruhanisnot/qwen-jaccard-test/qwen-jaccard-test/qwen_freq_cache.pkl"

if not os.path.exists(FREQ_CACHE):
    FREQ_CACHE = "qwen_freq_cache.pkl"

if os.path.exists(FREQ_CACHE):
    print(f"Loading frequency cache from {FREQ_CACHE}...")
    with open(FREQ_CACHE, "rb") as f:
        freq_data = pickle.load(f)

    # freq_cache keys are always "python"/"medical"/"math"/"creative"
    # (domain_names in config uses "code"/"medical" — different from cache keys)
    chosen = cfg.get("chosen_option", "A")
    if chosen == "A":
        freq_a, freq_b = freq_data["python"], freq_data["medical"]
        label_a, label_b = "python", "medical"
    else:
        freq_a, freq_b = freq_data["math"], freq_data["creative"]
        label_a, label_b = "math", "creative"

    has_freq = True
    print(f"Loaded: {label_a} vs {label_b} (chosen pair: {cfg.get('chosen_pair', label_a + ' vs ' + label_b)})")
else:
    print(f"Frequency cache not found at {FREQ_CACHE}.")
    print("Per-layer table will be skipped — run qwen-jaccard-diagnostic.ipynb first.")
    has_freq = False
    freq_a = freq_b = None
    label_a = label_b = None

Loading frequency cache from qwen_freq_cache.pkl...
Loaded: python vs medical (chosen pair: Python vs Medical)


In [4]:
N_LAYERS  = cfg.get("num_layers",  24)    # Qwen1.5-MoE has 24 layers
N_EXPERTS = cfg.get("num_experts", 60)    # 60 routing experts (shared expert excluded)
TOP_N     = cfg.get("top_n_jaccard", 15)  # top quartile of 60 = 15

if has_freq:
    def jaccard_topk(fa, fb, top_n=TOP_N):
        top_a = set(sorted(fa, key=fa.get, reverse=True)[:top_n])
        top_b = set(sorted(fb, key=fb.get, reverse=True)[:top_n])
        union = top_a | top_b
        return len(top_a & top_b) / len(union) if union else 0.0

    topk_scores = [jaccard_topk(freq_a[i], freq_b[i]) for i in range(N_LAYERS)]
    ranked = sorted(enumerate(topk_scores), key=lambda x: x[1], reverse=True)
    top3   = ranked[:3]
    top3_layers = {l for l, _ in top3}

    print(f"TABLE 2: Qwen1.5-MoE Top-K Jaccard by Layer ({cfg.get('chosen_pair', label_a + ' vs ' + label_b)})")
    print("-" * 60)
    print(f"  {'Layer':<10} {'Jaccard':>10}  Notes")
    print("-" * 60)
    for layer_idx, score in enumerate(topk_scores):
        marker = "  <- TOP-3" if layer_idx in top3_layers else ""
        if layer_idx == cfg.get("target_layer", -1):
            marker += "  * TARGET"
        print(f"  Layer {layer_idx:<4} {score:>10.4f}{marker}")
    print("-" * 60)
    print(f"  Mean (all layers): {sum(topk_scores)/len(topk_scores):.4f}")
    print()
    print("Top-3 highest-overlap layers:")
    for rank, (layer_idx, score) in enumerate(top3, 1):
        print(f"  #{rank}: Layer {layer_idx} -- Jaccard = {score:.4f}")
    print()
    print(f"Target layer:  {cfg.get('target_layer')} | target expert: {cfg.get('target_expert')}")
else:
    print("Frequency cache not available. To enable this table:")
    print("  1. Re-run qwen-jaccard-diagnostic.ipynb and download qwen_freq_cache.pkl")
    print("  2. Upload to Kaggle as a dataset input")
    print()
    print(f"Target layer  (from config): {cfg.get('target_layer')}")
    print(f"Target expert (from config): {cfg.get('target_expert')}")

TABLE 2: Qwen1.5-MoE Top-K Jaccard by Layer (Python vs Medical)
------------------------------------------------------------
  Layer         Jaccard  Notes
------------------------------------------------------------
  Layer 0        0.1538  <- TOP-3  * TARGET
  Layer 1        0.0714
  Layer 2        0.1111  <- TOP-3
  Layer 3        0.1111  <- TOP-3
  Layer 4        0.0714
  Layer 5        0.0345
  Layer 6        0.1111
  Layer 7        0.0345
  Layer 8        0.0714
  Layer 9        0.0714
  Layer 10       0.0000
  Layer 11       0.1111
  Layer 12       0.0000
  Layer 13       0.0345
  Layer 14       0.0345
  Layer 15       0.0345
  Layer 16       0.1111
  Layer 17       0.1111
  Layer 18       0.0345
  Layer 19       0.0714
  Layer 20       0.0714
  Layer 21       0.1111
  Layer 22       0.0345
  Layer 23       0.1111
------------------------------------------------------------
  Mean (all layers): 0.0714

Top-3 highest-overlap layers:
  #1: Layer 0 -- Jaccard = 0.1538
  #2: Layer 2

## Table 3 — Cross-Model Summary (for thesis Section 4.2 table)

In [5]:
# Chosen Jaccard for each model (the pair used in the actual experiment)
phi_chosen   = PHI_JACCARD_PYTHON_MEDICAL   # Phi used Python vs Medical
olmoe_chosen = OLMOE_JACCARD_MATH_CREATIVE  # OLMoE used Math vs Creative
qwen_chosen  = cfg["jaccard_" + ("python_medical" if cfg.get("chosen_option", "A") == "A" else "math_creative")]
qwen_null_v  = cfg["jaccard_null_baseline"]

# Phi null is ~1.0 (raw set, same domain ~ full overlap)
phi_sep_ratio   = phi_chosen   / 1.0
olmoe_sep_ratio = olmoe_chosen / max(OLMOE_NULL, 0.001)
qwen_sep_ratio  = qwen_chosen  / max(qwen_null_v, 0.001)

print("TABLE 3: Cross-Model Summary")
print("=" * 75)
summary_rows = [
    ("Metric",              "Phi-3.5-MoE",   "OLMoE-1B-7B",  "Qwen1.5-MoE"),
    ("-" * 24,              "-" * 14,          "-" * 14,         "-" * 14),
    ("Chosen pair",         "Py vs Med",       "Math vs Crtv",   cfg.get("chosen_pair", "Py vs Med")[:13]),
    ("Chosen Jaccard",      "%.3f" % phi_chosen, "%.3f" % olmoe_chosen, "%.3f" % qwen_chosen),
    ("Null baseline",       "~1.000",          "%.3f" % OLMOE_NULL,     "%.3f" % qwen_null_v),
    ("Separation ratio",    "%.3fx" % phi_sep_ratio, "%.3fx" % olmoe_sep_ratio, "%.3fx" % qwen_sep_ratio),
    ("Target layer",        "—",               "6",              str(cfg.get("target_layer", "?"))),
    ("Target expert",       "—",               "18",             str(cfg.get("target_expert", "?"))),
]
for r in summary_rows:
    print("%-26s %16s %16s %16s" % r)

print()
print("Note: Phi separation ratio uses raw-set Jaccard (same-domain ~ 1.0).")
print("      OLMoE and Qwen use top-K Jaccard with null baselines from held-out same-domain splits.")
print("      All three show strong domain separation (ratio < 0.15x), motivating expert specialisation.")

TABLE 3: Cross-Model Summary
Metric                          Phi-3.5-MoE      OLMoE-1B-7B      Qwen1.5-MoE
------------------------     --------------   --------------   --------------
Chosen pair                       Py vs Med     Math vs Crtv    Python vs Med
Chosen Jaccard                        0.056            0.053            0.071
Null baseline                        ~1.000            0.840            0.510
Separation ratio                     0.056x           0.063x           0.140x
Target layer                              —                6                0
Target expert                             —               18                5

Note: Phi separation ratio uses raw-set Jaccard (same-domain ~ 1.0).
      OLMoE and Qwen use top-K Jaccard with null baselines from held-out same-domain splits.
      All three show strong domain separation (ratio < 0.15x), motivating expert specialisation.


## Thesis Interpretation Note

In [6]:
qwen_chosen_j = cfg["jaccard_" + ("python_medical" if cfg.get("chosen_option", "A") == "A" else "math_creative")]
sep_ratio     = qwen_chosen_j / max(qwen_null_v, 0.001)

# Shared expert note
shared_note = (
    "Unlike Phi-3.5-MoE and OLMoE, Qwen1.5-MoE includes a permanently-active shared expert "
    "that fires on every token alongside the top-4 routing experts. "
    "Jaccard analysis targets only the 60 routing experts (shared expert excluded), "
    "consistent with the routing analysis in the other two models."
)

if qwen_chosen_j < 0.10:
    body = (
        f"Qwen1.5-MoE's routing is strongly domain-separated "
        f"(top-K Jaccard {qwen_chosen_j:.3f}, separation ratio {sep_ratio:.2f}×). "
        f"This is consistent with Phi-3.5-MoE (0.056) and OLMoE (0.053), confirming that "
        "domain specialisation in expert routing is a structural property shared across "
        "sparse MoE architectures. The Hierarchical ReLU-LoRA spawning mechanism, which "
        "targets gradient-space conflict rather than routing-space overlap, generalises "
        "cleanly to Qwen's distinct routing configuration (top-4 of 60, 6.7% uniform baseline)."
    )
elif qwen_chosen_j < 0.20:
    body = (
        f"Qwen1.5-MoE shows low routing overlap (top-K Jaccard {qwen_chosen_j:.3f}), "
        f"broadly consistent with Phi-3.5-MoE (0.056) and OLMoE (0.053). "
        f"The null baseline of {qwen_null_v:.3f} confirms genuine domain separation "
        f"({sep_ratio:.2f}× above coincidental overlap). "
        "Gradient entanglement arises in shared parameter space, not routing assignment — "
        "the Hierarchical ReLU-LoRA spawning mechanism addresses this directly and "
        "applies without modification to Qwen's top-4/60 routing scheme."
    )
else:
    body = (
        f"Qwen1.5-MoE shows moderate routing overlap (top-K Jaccard {qwen_chosen_j:.3f}), "
        f"higher than Phi-3.5-MoE (0.056) and OLMoE (0.053). "
        "This indicates more routing-level domain mixing in Qwen's top-4/60 configuration, "
        "potentially because a smaller relative top-k (6.7% vs 12.5%) concentrates certain "
        "domain signals on fewer experts. The higher overlap strengthens the motivation for "
        "hierarchical spawning — both routing-space and gradient-space conflict are present, "
        "making domain interference more acute and the benefit of independent sub-adapters "
        "more pronounced."
    )

print("THESIS NOTE (copy into Section 4.2 — Qwen paragraph):")
print("-" * 70)
print(body)
print()
print("SHARED EXPERT NOTE (optional footnote or methodological note):")
print("-" * 70)
print(shared_note)
print("-" * 70)
print()
print(f"Target layer for Qwen experiments: Layer {cfg.get('target_layer')}")
print(f"Target expert:                      Expert {cfg.get('target_expert')}")
print()
print("Pass these to qwen-smoke-test.ipynb (fallback defaults already set):")
print(f"  TARGET_LAYER  = {cfg.get('target_layer')}")
print(f"  TARGET_EXPERT = {cfg.get('target_expert')}")

THESIS NOTE (copy into Section 4.2 — Qwen paragraph):
----------------------------------------------------------------------
Qwen1.5-MoE's routing is strongly domain-separated (top-K Jaccard 0.071, separation ratio 0.14×). This is consistent with Phi-3.5-MoE (0.056) and OLMoE (0.053), confirming that domain specialisation in expert routing is a structural property shared across sparse MoE architectures. The Hierarchical ReLU-LoRA spawning mechanism, which targets gradient-space conflict rather than routing-space overlap, generalises cleanly to Qwen's distinct routing configuration (top-4 of 60, 6.7% uniform baseline).

SHARED EXPERT NOTE (optional footnote or methodological note):
----------------------------------------------------------------------
Unlike Phi-3.5-MoE and OLMoE, Qwen1.5-MoE includes a permanently-active shared expert that fires on every token alongside the top-4 routing experts. Jaccard analysis targets only the 60 routing experts (shared expert excluded), consistent 